# Régression

Le descriptif a montré que la pauvreté seule est presque sans lien avec l'obésité, et que la région pèse beaucoup. Ce notebook teste ce qu'il reste des liens une fois la région prise en compte.

Les résultats décrivent des associations entre départements, pas des causes.

## 1. Chargement

In [8]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv("../data/processed/departements.csv", dtype={"dep": str, "region": str})
df["log_densite"] = np.log(df["densite"])
df.shape

(96, 18)

On prend le logarithme de la densité : elle va de 15 à près de 20 000 hab/km², et sans le logarithme Paris dominerait le modèle.

## 2. Pauvreté seule, puis pauvreté avec la région

In [9]:
modele1 = smf.ols("obesite ~ taux_pauvrete", data=df).fit(cov_type="HC3")

resultats1 = pd.DataFrame({
    "coefficient": modele1.params,
    "ecart_type": modele1.bse,
    "p_valeur": modele1.pvalues,
})
print("R2 :", round(modele1.rsquared, 3))
resultats1.round(3)

R2 : 0.006


,coefficient,ecart_type,p_valeur
Intercept,14.610,1.510,0.000
taux_pauvrete,0.083,0.102,0.418


**Interprétation :** sans la région, le coefficient de la pauvreté est de 0,08 point d'obésité par point de pauvreté, mais il n'est pas significatif (p = 0,42) et le R² est de 0,6 % : la pauvreté seule n'explique presque rien. C'est cohérent avec la corrélation de 0,10 du descriptif.

On ajoute maintenant la région comme effet fixe (`C(region)`), c'est-à-dire qu'on compare les départements au sein de leur région. Les écarts-types sont robustes (`HC3`), c'est-à-dire moins sensibles aux départements atypiques.

In [10]:
modele2 = smf.ols("obesite ~ taux_pauvrete + C(region)", data=df).fit(cov_type="HC3")

resultats2 = pd.DataFrame({
    "coefficient": modele2.params,
    "ecart_type": modele2.bse,
    "p_valeur": modele2.pvalues,
})
print("R2 :", round(modele2.rsquared, 3))
resultats2.loc[["taux_pauvrete"]].round(3)

R2 : 0.589


,coefficient,ecart_type,p_valeur
taux_pauvrete,0.316,0.107,0.003


In [11]:
modele_region = smf.ols("obesite ~ C(region)", data=df).fit()
print("R2 de la région seule :", round(modele_region.rsquared, 3))
print("R2 ajusté du modèle avec pauvreté et région :", round(modele2.rsquared_adj, 3))
print("R2 ajusté de la région seule :", round(modele_region.rsquared_adj, 3))

R2 de la région seule : 0.535
R2 ajusté du modèle avec pauvreté et région : 0.524
R2 ajusté de la région seule : 0.468


**Interprétation :** avec la région, le coefficient de la pauvreté passe à 0,32 point d'obésité par point de pauvreté. Il est significatif (p = 0,003, intervalle de confiance à 95 % de 0,11 à 0,53). À région égale, un département qui a 1 point de pauvreté de plus a en moyenne 0,3 point d'obésité de plus. Le lien n'apparaissait pas dans le descriptif parce que la région le masquait.

Le R² monte à 0,59, mais la région seule en explique déjà 0,54 : la région fait presque tout le travail, et la pauvreté ajoute environ 5 points de R². Son effet est réel mais modeste.